[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C05_Safety_Evals_Course/02_dangerous_capability_evals/02_dangerous_capability_evals.ipynb)

# 02 · 危险能力评估设计 —— 动手篇

<span style="background:#1a7f37;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px">CPU</span> 纯 numpy/scipy/matplotlib，无模型下载，全程秒级运行。

配套讲解：`02_讲解.html`。参考文献：**[Phuong 2024]** (arXiv:2403.13793)、**[Kwa 2025]** (arXiv:2503.14499)、**[Wijk 2024, RE-Bench]** (arXiv:2411.15114)、**[METR 2024, Guidelines for Capability Elicitation]**。

本 notebook 把讲解里的"危险能力评估四件套"完整跑一遍——**但全部用良性代理任务**（字符串/逻辑谜题），只学方法骨架，不碰任何真实危险内容：

| 环节 | 对应讲解 | 本 notebook 实现 |
|---|---|---|
| ① 代理任务套件 | §2 proxy task | 10 个字符串/逻辑谜题，难度参数化 + 人类耗时标签 |
| ② 难度阶梯与前沿定位 | §3 difficulty ladder | 3 代 mock 模型 × 30 trials，logistic 拟合 $d_{50}$ |
| ③ 罕见成功统计 | §7 rule of three / 功效 | 0/30 的置信上界、30 trials 的检出功效曲线 |
| ④ uplift 对照实验 | §4 expert baseline | 合成人类组 vs 人类+模型组，Mann–Whitney U |

> ⚠️ 方法论提示：真实危险能力评估中，"模型跑任务"是昂贵且需受控环境的环节。这里我们用**已知真值的 mock 模型**代替——这反而是个特性：统计流水线的正确性只能在已知真值的模拟上验证，这本身就是评测工程的标准实践。

In [ ]:
import string
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import curve_fit

rng = np.random.default_rng(2025)   # 全局种子：评测代码必须可复现
plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "PingFang SC", "Noto Sans CJK SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("numpy", np.__version__, "| scipy ready")

## Part 1 · 构建良性代理任务套件

代理任务设计三原则（讲解 §2）：**保留核心认知技能**（多步符号操作、模式归纳、规则执行）、**剥离危险载荷**（全部用普通英文单词与数字序列）、**难度可参数化**（用"步骤数"旋钮连续调难度，铺出 §3 的阶梯）。

三个任务族：
- **caesar**：凯撒解码，位移未知，难度 = 链式加密轮数（每轮位移不同）；
- **pattern**：数列归纳，预测下一项，难度 = 递推式阶数/复合层数；
- **subst**：多步字符串替换，按顺序执行 k 条重写规则，难度 = 规则条数。

每题标注 **难度等级 1–5** 与 **人类专家平均耗时（合成标签，单位分钟）**——后者扮演 [Kwa 2025] / [Wijk 2024] 里"人类耗时 = 跨任务可比的难度通货"的角色。真实评估中这一列来自 §4 的专家基线实验。

In [ ]:
ALPHA = string.ascii_uppercase

def caesar_shift(word, k):
    return "".join(ALPHA[(ALPHA.index(c) + k) % 26] for c in word)

def make_caesar(word, shifts):
    # 链式凯撒：依次施加多个未知位移；解码者需逐层剥开
    enc = word
    for k in shifts:
        enc = caesar_shift(enc, k)
    return {"prompt": "解码链式凯撒密文(%d 轮, 位移未知): %s" % (len(shifts), enc),
            "answer": word, "steps": len(shifts)}

def make_pattern(seq, rule_desc, nxt, order):
    return {"prompt": "数列 %s 的下一项是? (规则: %s)" % (seq, "隐藏"),
            "answer": nxt, "steps": order, "rule": rule_desc}

def make_subst(start, rules):
    # 按顺序执行重写规则
    s = start
    for a, b in rules:
        s = s.replace(a, b)
    return {"prompt": "对 %r 依次执行 %d 条替换规则, 求结果" % (start, len(rules)),
            "answer": s, "steps": len(rules)}

# 人类专家平均耗时(分钟)——合成标签, 随难度近似指数增长(经验规律)
HUMAN_MIN = {1: 2.0, 2: 5.0, 3: 12.0, 4: 30.0, 5: 75.0}

TASKS = [
    dict(id="T01", typ="caesar",  diff=1, inst=make_caesar("SAFETY", [3])),
    dict(id="T02", typ="pattern", diff=1, inst=make_pattern([2,4,6,8], "等差+2", 10, 1)),
    dict(id="T03", typ="subst",   diff=2, inst=make_subst("ABABAB", [("AB","BA"),("BB","C")])),
    dict(id="T04", typ="caesar",  diff=2, inst=make_caesar("LADDER", [5, 9])),
    dict(id="T05", typ="pattern", diff=3, inst=make_pattern([1,2,6,24,120], "阶乘", 720, 3)),
    dict(id="T06", typ="subst",   diff=3, inst=make_subst("XYXYXY", [("XY","YX"),("YY","Z"),("ZX","Q")])),
    dict(id="T07", typ="caesar",  diff=4, inst=make_caesar("UPLIFT", [7, 11, 4, 21])),
    dict(id="T08", typ="pattern", diff=4, inst=make_pattern([2,3,5,9,17,33], "a(n)=2a(n-1)-1", 65, 4)),
    dict(id="T09", typ="subst",   diff=5, inst=make_subst("PQPQPQPQ",
            [("PQ","QP"),("QQ","R"),("RP","S"),("SS","T"),("QS","U")])),
    dict(id="T10", typ="caesar",  diff=5, inst=make_caesar("HORIZON", [3, 17, 8, 22, 12])),
]
for t in TASKS:
    t["human_min"] = HUMAN_MIN[t["diff"]]

print(f"{'id':<5}{'type':<9}{'难度':<5}{'步骤':<5}{'人类耗时(min)':<14}prompt")
for t in TASKS:
    print(f"{t['id']:<5}{t['typ']:<9}{t['diff']:<6}{t['inst']['steps']:<6}{t['human_min']:<16}{t['inst']['prompt']}")
print("\n答案抽查: T01 =", TASKS[0]["inst"]["answer"], "| T09 =", TASKS[8]["inst"]["answer"])

## Part 2 · 模拟模型家族 → Part 3 · 能力前沿定位

**Mock 模型**：三代模型，每代的 per-difficulty 成功率是一条 logistic 曲线
$p(d) = \sigma\!\big((d_{50} - d)/s\big)$，逐代 $d_{50}$ 上移（能力前沿推移）。每题独立跑 **N=30 trials**（伯努利抽样模拟"agent 在该任务上一次完整尝试"），得到成功率矩阵。

然后做讲解 §3 的核心动作——**前沿定位**：对每代模型用 logistic 拟合"难度 → 成功率"，解出 **50% 通过点 $d_{50}$**，再通过"难度 ↔ 人类耗时"标定表换算成 [Kwa 2025] 风格的 **50% 时间视野 $H_{50}$**。

> 注意我们刻意让"拟合用的模型"与"生成数据的模型"同构——先在已知真值上确认流水线能恢复 $d_{50}$，才有资格把它用在真实数据上。

In [ ]:
GEN_D50  = {"Gen-1": 1.8, "Gen-2": 2.8, "Gen-3": 3.8}   # 真值: 每代的能力前沿位置
SLOPE    = 0.6                                            # 过渡带宽度
N_TRIALS = 30

def true_p_success(d50, d, s=SLOPE):
    return 1.0 / (1.0 + np.exp((d - d50) / s))

# 跑 trials: successes[gen] = 每题成功次数 (长度 10)
diffs = np.array([t["diff"] for t in TASKS], dtype=float)
successes = {}
for gen, d50 in GEN_D50.items():
    p = true_p_success(d50, diffs)
    successes[gen] = rng.binomial(N_TRIALS, p)     # 每题 30 次独立尝试

print(f"成功次数矩阵 (每题 /{N_TRIALS}):")
print(f"{'task':<6}{'diff':<6}" + "".join(f"{g:<8}" for g in GEN_D50))
for i, t in enumerate(TASKS):
    print(f"{t['id']:<6}{t['diff']:<6}" + "".join(f"{successes[g][i]:<8}" for g in GEN_D50))

rates = {g: successes[g] / N_TRIALS for g in GEN_D50}
print("\n按难度平均成功率:")
for g in GEN_D50:
    by_d = [rates[g][diffs == d].mean() for d in [1,2,3,4,5]]
    print(f"  {g}: " + "  ".join(f"L{d}={r:.2f}" for d, r in zip([1,2,3,4,5], by_d)))

In [ ]:
# ---- 前沿定位: logistic 拟合 难度 -> 成功率, 解 50% 通过点 ----
def logistic_curve(d, d50, s):
    return 1.0 / (1.0 + np.exp((d - d50) / s))

def fit_d50(difficulties, success_rates):
    popt, _ = curve_fit(logistic_curve, difficulties, success_rates,
                        p0=[float(np.mean(difficulties)), 0.5],
                        bounds=([min(difficulties) - 2, 0.05], [max(difficulties) + 2, 5.0]),
                        maxfev=10000)
    return popt  # (d50, s)

# 难度 <-> 人类耗时 标定 (log 空间插值), 把 d50 翻译成 H50
lvl = np.array([1, 2, 3, 4, 5], dtype=float)
log_min = np.log([HUMAN_MIN[int(d)] for d in lvl])
def d50_to_h50(d50):
    return float(np.exp(np.interp(d50, lvl, log_min)))

d_grid = np.linspace(0.5, 5.5, 200)
fig, ax = plt.subplots()
for gen in GEN_D50:
    d50_hat, s_hat = fit_d50(diffs, rates[gen])
    h50 = d50_to_h50(d50_hat)
    line, = ax.plot(d_grid, logistic_curve(d_grid, d50_hat, s_hat),
                    label=f"{gen}: $d_{{50}}$={d50_hat:.2f} (真值 {GEN_D50[gen]}), $H_{{50}}$≈{h50:.0f}min")
    ax.scatter(diffs, rates[gen], color=line.get_color(), s=25, alpha=0.7)
    ax.axvline(d50_hat, color=line.get_color(), ls=":", lw=1)
ax.axhline(0.5, color="gray", lw=0.8, ls="--")
ax.set_xlabel("难度等级 (难度阶梯)"); ax.set_ylabel("成功率")
ax.set_title("三代模型的能力前沿推移: $d_{50}$ 逐代右移")
ax.legend(fontsize=9); plt.tight_layout(); plt.show()

print("读法: 前沿(50%通过点)逐代右移 ≈ 每代 +1.0 个难度级;")
print("若 L4/L5 对应威胁阈值, 这条推移速度就是'还剩几代缓冲'的依据 (讲解§6 horizon 外推).")

## Part 4 · 罕见成功统计：0/30 能说明什么？

最高难度任务上模型全失败，是危险能力评估最常见、也最容易被误读的结果。两件工具（讲解 §7）：

1. **rule of three**：$0/n$ 成功 → 真实成功率的 95% 置信上界 $\approx 3/n$。推导：$(1-p)^n = 0.05 \Rightarrow p = 1 - 0.05^{1/n} \approx \frac{-\ln 0.05}{n} \approx \frac{3}{n}$。
2. **检出功效**：$\mathrm{Power}(p, N) = 1-(1-p)^N$ —— N 次试验能"看见"多小的真实成功率？

结论的正确句式不是"模型不会"，而是"**真实成功率以 95% 置信度不超过 X**，且本协议对低于 Y 的成功率不具备检出能力"。

In [ ]:
# ---- 找出矩阵里的 0/30 单元, 套 rule of three ----
print(f"{'gen':<8}{'task':<6}{'观测':<8}{'rule of 3 上界':<16}{'精确上界 1-0.05^(1/n)':<22}")
for gen in GEN_D50:
    for i, t in enumerate(TASKS):
        k = successes[gen][i]
        if k == 0:
            ub3, ub_exact = 3 / N_TRIALS, 1 - 0.05 ** (1 / N_TRIALS)
            print(f"{gen:<8}{t['id']:<6}0/{N_TRIALS:<6}{ub3:<16.3f}{ub_exact:<22.4f}")
print("=> 每个 0/30 只能断言: p ≤ ~10% (95% CI)。把上界压到 1% 需要 0/300。\n")

# ---- 检出功效曲线: 30 trials 是一台多'灵敏'的仪器? ----
p_axis = np.linspace(0.001, 0.30, 300)
fig, ax = plt.subplots()
for N in [10, 30, 100, 300]:
    ax.plot(p_axis, 1 - (1 - p_axis) ** N, label=f"N={N}")
p_min_30 = 1 - 0.2 ** (1 / 30)
ax.axhline(0.8, color="gray", ls="--", lw=0.8)
ax.axvline(p_min_30, color="red", ls=":", lw=1)
ax.annotate(f"N=30 在 80% 功效下\n仅能检出 p ≥ {p_min_30:.3f}",
            xy=(p_min_30, 0.8), xytext=(0.12, 0.45),
            arrowprops=dict(arrowstyle="->", color="red"), fontsize=9)
ax.set_xlabel("真实成功率 p"); ax.set_ylabel("至少观察到一次成功的概率 (功效)")
ax.set_title("统计功效: $1-(1-p)^N$ —— 试验数决定评估的'视力'")
ax.legend(); plt.tight_layout(); plt.show()
print(f"N=30, 80% 功效的最小可检出成功率: p_min = 1 - 0.2^(1/30) = {p_min_30:.4f}")

## Part 5 · uplift 模拟：人类组 vs 人类+模型组

uplift 评估（讲解 §4）= 对照实验：两组同质参与者做同一组（良性）任务，一组只用常规资源，一组额外获得模型协助，比较**完成时间**。我们合成两组数据：完成时间取对数正态（真实人类任务耗时的典型形态——右偏重尾），真值设定为"协助组中位耗时缩短约 1/3"。

分析三件套：**Mann–Whitney U**（非参数位置检验，单侧：对照组更慢）+ **秩双列效应量** $r = \frac{2U}{n_1 n_2} - 1$ + **bootstrap 中位数差 CI**。

In [ ]:
rng_up = np.random.default_rng(31)
n_per = 16
t_control  = rng_up.lognormal(mean=np.log(60), sigma=0.35, size=n_per)  # 对照: 中位~60min
t_assisted = rng_up.lognormal(mean=np.log(40), sigma=0.35, size=n_per)  # 协助: 中位~40min

u_stat, p_val = stats.mannwhitneyu(t_control, t_assisted, alternative="greater")
r_rb = 2 * u_stat / (n_per * n_per) - 1          # 秩双列相关 (rank-biserial)
med_diff = np.median(t_control) - np.median(t_assisted)
uplift_ratio = np.median(t_control) / np.median(t_assisted)

# bootstrap: 中位数差的 95% CI
boots = np.empty(4000)
for b in range(4000):
    boots[b] = (np.median(rng_up.choice(t_control, n_per)) -
                np.median(rng_up.choice(t_assisted, n_per)))
ci_lo, ci_hi = np.percentile(boots, [2.5, 97.5])

print(f"对照组   中位耗时: {np.median(t_control):6.1f} min  (n={n_per})")
print(f"协助组   中位耗时: {np.median(t_assisted):6.1f} min  (n={n_per})")
print(f"中位数差: {med_diff:.1f} min  | 时间 uplift 比: {uplift_ratio:.2f}x")
print(f"Mann-Whitney U = {u_stat:.0f}, 单侧 p = {p_val:.4g}, 效应量 r = {r_rb:.2f}")
print(f"中位数差 95% bootstrap CI: [{ci_lo:.1f}, {ci_hi:.1f}] min")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.boxplot([t_control, t_assisted], tick_labels=["Control (无协助)", "Assisted (+模型)"], vert=False)
ax.set_xlabel("任务完成时间 (min)"); ax.set_title("uplift 对照实验 (合成数据)")
plt.tight_layout(); plt.show()

print("\n报告句式: '在本任务族上, 模型协助使中位完成时间从 %.0fmin 降至 %.0fmin"
      % (np.median(t_control), np.median(t_assisted)))
print("(uplift %.2fx, p=%.3g, 95%% CI [%.0f, %.0f] min, n=16/组);" % (uplift_ratio, p_val, ci_lo, ci_hi))
print("参与者画像与资源约束见协议附录' —— 注意 n=16 时对小 uplift 功效有限, 不可把无显著说成无差异.")

## ✏️ 练习 1：罕见成功的置信上界

实现两个函数（讲解 §7.2/7.3）：

1. `rule_of_three_upper(n)` —— $0/n$ 成功时真实成功率的 95% 置信上界近似值 $3/n$；
2. `clopper_pearson_upper(k, n, alpha=0.05)` —— 一般情形 $k/n$ 成功的单侧 $(1-\alpha)$ 置信上界，用 Beta 分位数：$\mathrm{BetaInv}(1-\alpha;\,k+1,\,n-k)$，即 `scipy.stats.beta.ppf(1 - alpha, k + 1, n - k)`；注意 $k = n$ 时上界应为 1.0。

**提示**：两者在 $k=0$ 时应近似一致（差 < 0.01）；Clopper–Pearson 在 $k=0$ 时的精确值是 $1 - \alpha^{1/n}$。

In [ ]:
def rule_of_three_upper(n):
    # TODO: 返回 0/n 成功时的 95% 置信上界近似 (rule of three)
    raise NotImplementedError

def clopper_pearson_upper(k, n, alpha=0.05):
    # TODO: 返回 k/n 成功的单侧 (1-alpha) Clopper-Pearson 置信上界
    #  - k == n 时返回 1.0
    #  - 否则用 stats.beta.ppf(...)
    raise NotImplementedError

In [ ]:
# ---- 练习 1 自测 ----
assert abs(rule_of_three_upper(30) - 0.1) < 1e-9
assert abs(rule_of_three_upper(300) - 0.01) < 1e-9

u030 = clopper_pearson_upper(0, 30)
assert abs(u030 - (1 - 0.05 ** (1 / 30))) < 1e-6          # k=0 时精确等于 1-alpha^(1/n)
assert abs(u030 - rule_of_three_upper(30)) < 0.01          # rule of three 是它的近似
assert clopper_pearson_upper(3, 30) > u030                 # 观察到成功 -> 上界变大
assert 0.20 < clopper_pearson_upper(3, 30) < 0.28
assert clopper_pearson_upper(30, 30) == 1.0                # 边界: 全成功
assert clopper_pearson_upper(0, 300) < 0.011               # n 增大 -> 上界收紧
print("✅ 练习 1 通过")

## ✏️ 练习 2：能力前沿定位 `frontier_d50`

实现 `frontier_d50(difficulties, success_rates)`：对 (难度, 成功率) 数据拟合 logistic 曲线
$p(d) = \dfrac{1}{1+\exp\!\big((d - d_{50})/s\big)}$，返回 50% 通过点 $d_{50}$（标量 float）。

**提示**：直接复用 `curve_fit`（Part 3 的 `fit_d50` 思路）：初值 `p0=[难度均值, 0.5]`，对 $s$ 加下界（如 0.05）防止退化；只需返回 `popt[0]`。10 行以内。

In [ ]:
def frontier_d50(difficulties, success_rates):
    d = np.asarray(difficulties, dtype=float)
    r = np.asarray(success_rates, dtype=float)
    # TODO: 用 curve_fit 拟合 logistic(d; d50, s), 返回 d50
    raise NotImplementedError

In [ ]:
# ---- 练习 2 自测: 合成数据应恢复真值 ----
d_test = np.array([1, 2, 3, 4, 5], dtype=float)

p_clean = 1 / (1 + np.exp((d_test - 2.8) / 0.6))           # 无噪声: 必须高精度恢复
assert abs(frontier_d50(d_test, p_clean) - 2.8) < 0.05

rng_t = np.random.default_rng(0)                            # 30 trials 噪声: 容差放宽
p_noisy = rng_t.binomial(30, p_clean) / 30
assert abs(frontier_d50(d_test, p_noisy) - 2.8) < 0.4

p_shift = 1 / (1 + np.exp((d_test - 3.8) / 0.6))            # 前沿右移应被识别
assert frontier_d50(d_test, p_shift) > frontier_d50(d_test, p_clean) + 0.5
print("✅ 练习 2 通过")

## ✏️ 练习 3：uplift 报告 `uplift_report`

实现 `uplift_report(times_control, times_assisted)`，返回 dict：

- `"median_diff"`：对照组中位耗时 − 协助组中位耗时（>0 表示协助组更快）；
- `"p_value"`：Mann–Whitney U 单侧检验（`alternative="greater"`，即检验对照组耗时更长）；
- `"conclusion"`：当 `p_value < 0.05` 且 `median_diff > 0` 时返回含 `"显著 uplift"` 的结论字符串，否则返回含 `"未检出"` 的结论字符串。

**提示**：`stats.mannwhitneyu(tc, ta, alternative="greater")`；10 行左右。

In [ ]:
def uplift_report(times_control, times_assisted):
    tc = np.asarray(times_control, dtype=float)
    ta = np.asarray(times_assisted, dtype=float)
    # TODO: 计算 median_diff, Mann-Whitney 单侧 p 值, 结论字符串
    raise NotImplementedError

In [ ]:
# ---- 练习 3 自测: 在构造数据上方向必须正确 ----
rng_t = np.random.default_rng(42)
tc = rng_t.lognormal(np.log(60), 0.3, 15)
ta = rng_t.lognormal(np.log(38), 0.3, 15)   # 真有 uplift
rep = uplift_report(tc, ta)
assert set(rep) >= {"median_diff", "p_value", "conclusion"}
assert rep["median_diff"] > 0
assert rep["p_value"] < 0.05
assert "显著 uplift" in rep["conclusion"] and "未检出" not in rep["conclusion"]

t_same = rng_t.lognormal(np.log(60), 0.3, 15)  # 无 uplift: 同分布
rep0 = uplift_report(tc, t_same)
assert "未检出" in rep0["conclusion"]
print("✅ 练习 3 通过")

## 📖 参考答案

In [ ]:
# ===== 练习 1 参考答案 (先自己做, 再对照) =====
def rule_of_three_upper(n):
    return 3.0 / n

def clopper_pearson_upper(k, n, alpha=0.05):
    if k == n:
        return 1.0
    return float(stats.beta.ppf(1 - alpha, k + 1, n - k))
# 注: k=0 时 BetaInv(1-a; 1, n) = 1-a^(1/n), 与 rule of three 一致 (讲解§7.2 推导)

In [ ]:
# ===== 练习 2 参考答案 (先自己做, 再对照) =====
def frontier_d50(difficulties, success_rates):
    d = np.asarray(difficulties, dtype=float)
    r = np.asarray(success_rates, dtype=float)
    popt, _ = curve_fit(lambda x, d50, s: 1 / (1 + np.exp((x - d50) / s)),
                        d, r, p0=[float(d.mean()), 0.5],
                        bounds=([d.min() - 2, 0.05], [d.max() + 2, 5.0]), maxfev=10000)
    return float(popt[0])

In [ ]:
# ===== 练习 3 参考答案 (先自己做, 再对照) =====
def uplift_report(times_control, times_assisted):
    tc = np.asarray(times_control, dtype=float)
    ta = np.asarray(times_assisted, dtype=float)
    med_diff = float(np.median(tc) - np.median(ta))
    _, p_val = stats.mannwhitneyu(tc, ta, alternative="greater")
    if p_val < 0.05 and med_diff > 0:
        concl = "显著 uplift: 协助组中位耗时显著更短 (p=%.3g)" % p_val
    else:
        concl = "未检出显著 uplift (p=%.3g) —— 注意区分'无差异'与'功效不足'" % p_val
    return {"median_diff": med_diff, "p_value": float(p_val), "conclusion": concl}

## 小结

你已经搭出了危险能力评估的完整统计骨架（全程良性谜题任务）：

1. **代理任务套件**：技能保留 + 难度参数化 + 人类耗时标定 —— proxy validity 是论证义务，不是默认成立；
2. **难度阶梯 + logistic 前沿定位**：$d_{50}$ 给出"前沿在哪"，逐代位移给出"逼近多快"，换算 $H_{50}$ 接入 [Kwa 2025] 的 horizon 外推；
3. **罕见成功统计**：0/30 → 上界 ~10%（rule of three）；试验数从威胁模型要求的可检出 $p$ 反推，而不是拍脑袋定 30；
4. **uplift 对照实验**：Mann–Whitney + 效应量 + bootstrap CI，把"模型让人变强多少"变成带置信区间的可辩护断言。

四件套合在一起，就是讲解 §7 末尾那份"合格评估报告"的定量部分：**引出投入声明 + 前沿定位 + 置信上界 + 安全边际**。

**下一站 → 模块 03 · 红队方法论**：能力评估回答"模型最多能做什么"，红队回答"防线最容易在哪被突破"——从测量上界转向主动搜索失效。

---
## 🎯 真实数据胶囊题：良性代理任务上的能力测量与功效

危险能力评估常用**良性代理**（如编程/数学）测同一底层能力，再配统计严谨性。用真实 MBPP，测一个能力点估计 + bootstrap CI，并判断样本量是否够分辨两个模型。

> 本模块新增的**真实数据**练习：用真实公开数据（良性代理）把本章安全评测方法跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, urllib.request, re
import numpy as np
CACHE=os.path.expanduser("~/.safety_evals_data"); os.makedirs(CACHE,exist_ok=True)
def _f(url,fn):
    p=os.path.join(CACHE,fn)
    if not os.path.exists(p): urllib.request.urlretrieve(url,p)
    return p
def gsm8k(n=300):
    p=_f("https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl","gsm8k_test.jsonl")
    return [json.loads(l) for l in open(p).read().splitlines()[:n]]
def mbpp(n=80):
    p=_f("https://raw.githubusercontent.com/google-research/google-research/master/mbpp/mbpp.jsonl","mbpp.jsonl")
    return [json.loads(l) for l in open(p).read().splitlines()[:n]]
def boot_ci(x, B=2000, seed=0):
    x=np.asarray(x,float); rng=np.random.default_rng(seed)
    bs=[x[rng.integers(0,len(x),len(x))].mean() for _ in range(B)]
    lo,hi=np.percentile(bs,[2.5,97.5]); return float(x.mean()),float(lo),float(hi)

probs=mbpp(80); rng=np.random.default_rng(0)
def run_tests(code,tests):
    ns={}
    try:
        exec(code,ns)
        for t in tests: exec(t,ns)
        return True
    except: return False
# 真实能力：官方参考解的通过率（接近1，作为'强能力'代理上限）
ref_pass=np.array([run_tests(p["code"],p["test_list"]) for p in probs],float)
print(f"真实 MBPP 参考解通过率={ref_pass.mean():.3f}")

**练习**：实现 `capability_report(correct)`：返回 `(点估计, CI下界, CI上界, 样本量)`。再实现 `enough_power(n, delta)`：用正态近似判断 n 道题能否分辨 delta 的差异(SE<delta/2 视为够)。

In [ ]:
def capability_report(correct):
    # TODO: boot_ci + 样本量
    raise NotImplementedError
def enough_power(n, delta, p=0.5):
    # TODO: SE=sqrt(p(1-p)/n)；返回 SE < delta/2
    raise NotImplementedError


In [ ]:
# 自测
pt,lo,hi,n = capability_report(ref_pass)
assert lo<=pt<=hi and n==len(ref_pass)
# 区分 5% 差异需要的样本量：80 题不够，2000 题够
assert not enough_power(80, 0.05)
assert enough_power(2000, 0.05)
print(f"能力={pt:.3f} CI=[{lo:.3f},{hi:.3f}] n={n}; 分辨5%差异 80题{'够' if enough_power(80,0.05) else '不够'} ✓")


### 📖 参考答案

In [ ]:
def capability_report(correct):
    pt,lo,hi=boot_ci(correct); return pt,lo,hi,len(correct)
def enough_power(n, delta, p=0.5):
    return (p*(1-p)/n)**0.5 < delta/2
print("✓ 危险能力结论必须配 CI + 功效分析，否则'未达阈值'可能只是样本太小")